In [1]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

INFO: pip is looking at multiple versions of langchain-experimental to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 1.7 MB/s eta 0:00:03
   ---------- ----------------------------- 1.0/4.0 MB 1.9 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.0 MB 2.1 MB/s eta 0:00:02
   ----------------------- ---------------- 2.4/4.0 MB 2.4 MB/s eta 0:00:01
   ---------------------------- ----------- 2.9/4.0 MB 2.5 MB/s eta 0:00:01
   -------------------------------------- - 3.9/4.0 MB 2.8 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 2.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ------------- -------------------------- 1.0/3.1 MB 4.6 MB/s eta 0:00:01
   -----------------------

Built-in Tool - DuckDuckGo Search

In [3]:
!pip install -U ddgs


  Using cached ddgs-9.9.1-py3-none-any.whl.metadata (19 kB)
  Using cached fake_useragent-2.2.0-py3-none-any.whl.metadata (17 kB)
  Using cached h2-4.3.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached socksio-1.0.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached hpack-4.1.0-py3-none-any.whl.metadata (4.6 kB)
Using cached ddgs-9.9.1-py3-none-any.whl (41 kB)
Using cached fake_useragent-2.2.0-py3-none-any.whl (161 kB)
Using cached socksio-1.0.0-py3-none-any.whl (12 kB)
Using cached h2-4.3.0-py3-none-any.whl (61 kB)
Using cached hpack-4.1.0-py3-none-any.whl (34 kB)
Using cached hyperframe-6.1.0-py3-none-any.whl (13 kB)


In [6]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in india today')

print(results)

Get the latest updates and breaking news from India , including politics, elections, government, business, technology, and Bollywood. 1 day ago · Read all latest India news and get today 's breaking news updates on Indian politics, elections, sports, business, economy, cricket, current affairs, education, results and more on Oneindia. 5 days ago · The blast happened near the Red Fort metro station in a densely populated area of the Indian capital. Check today ' s top 10 news stories, headline news , breaking news , latest news , politics news, sports news, entertainment news and business news on Times of India Read full articles, watch videos, browse thousands of titles and more on the ' India ' topic with Google News.


In [8]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


Built-in Tool - Shell Tool

In [9]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('ls')

print(results)

Executing command:
 ls
'ls' is not recognized as an internal or external command,
operable program or batch file.



c:\Users\Lenovo\anaconda3\envs\amar\Lib\site-packages\langchain_community\tools\shell\tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


Custom Tools

In [10]:
from langchain_core.tools import tool
# Step 1 - create a function

def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [11]:
# Step 2 - add type hints

def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [12]:
# Step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [13]:
result = multiply.invoke({"a":3, "b":5})

In [14]:
print(result)

15


In [15]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [16]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


Method 2 - Using StructuredTool

In [17]:
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

In [18]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

In [19]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [20]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [21]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


Method 3 - Using BaseTool Class

In [23]:
from langchain.tools import BaseTool
from typing import Type

In [24]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

In [25]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [26]:
multiply_tool = MultiplyTool()

In [27]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


Toolkit

In [28]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [29]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [30]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)

add => Add two numbers
multiply => Multiply two numbers
